# 03 — LHP with PE-Core-G14-448 (LoRA fine-tune, A100 80GB)

**Upgrade 3 của plan**: thay BEiT-3 base trong LHP (Local-global Hybrid Perspective) bằng PE-Core-G14-448, fine-tune bằng **LoRA-only** (freeze 100% backbone, adapter chỉ vào Q/K/V của MHA).

**Why LoRA:** PE-G14 1.8B params, full fine-tune trên 1M synthetic captions sẽ catastrophic-forget kiến thức zero-shot. LoRA r=16 chỉ train ~8-12M params (~0.5%), giữ alignment, train siêu nhanh.

**LHP module:** probabilistic perspective sampling — `r ~ N(0.5, 1+δ)`, `δ∈[0,6]`. r>0.5 → local transform (anatomical crop driven by ViTPose keypoints từ 01b); else → global transform (PE-G14 augment chain).

**Hyperparams:**
- Image size 448 (PE-G14 native)
- Batch 128 (no grad-accum needed w/ LoRA)
- 3 epoch (paper-faithful)
- AdamW lr=3e-4 (LoRA-friendly, 50× higher than full FT)
- ITC contrastive loss only

**Inputs:**
- `manifests/train_manifest_trainable.parquet`
- `pose/vitposepp/train/chunk_*.npz` (keypoints cho local crop)
- `pose/vitposepp/gallery.npz` (optional)

**Outputs:**
- `checkpoints/lhp_peg14/lora_best/` — LoRA adapter (~40MB)
- `features/lhp_peg14/{gallery,queries,val,val_zs}.npz`
- `features/lhp_peg14/scores_lhp.pt`

**ETA:** ~2-3h trên A100 80GB BF16.

In [ ]:
# === Bootstrap & paths ===
from pathlib import Path
import os, sys, json, math, time, shutil, subprocess, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
    l2_normalize_np, save_npz_atomic,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']
MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']
OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']

LHP_CKPT_DIR = OUTPUT_ROOT / 'checkpoints' / 'lhp_peg14'
LHP_FEAT_DIR = OUTPUT_ROOT / 'features' / 'lhp_peg14'
LHP_CKPT_DIR.mkdir(parents=True, exist_ok=True)
LHP_FEAT_DIR.mkdir(parents=True, exist_ok=True)

device = select_a100_device()
print('LHP_CKPT_DIR:', LHP_CKPT_DIR)
print('LHP_FEAT_DIR:', LHP_FEAT_DIR)

In [ ]:
# === Install deps ===
def pip_install(*p):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *p])

try:
    import peft  # noqa: F401
except Exception:
    pip_install('peft>=0.13.0', 'safetensors')

try:
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as pe_transforms
except Exception:
    repo = NOTEBOOK_DIR / 'perception_models'
    if not repo.exists():
        subprocess.check_call([
            'git', 'clone',
            'https://github.com/facebookresearch/perception_models.git',
            str(repo),
        ])
    pip_install('-e', str(repo))
    if str(repo) not in sys.path:
        sys.path.append(str(repo))
    import core.vision_encoder.pe as pe
    import core.vision_encoder.transforms as pe_transforms

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from PIL import Image
from tqdm.auto import tqdm
import peft
print('peft:', peft.__version__, 'torch:', torch.__version__)

In [ ]:
# === Hyperparams (A100-80GB tuned + LoRA) ===
PE_MODEL_NAME = 'PE-Core-G14-448'
IMAGE_SIZE = 448
BATCH = 128                   # LoRA giảm act memory → no need grad-accum
NUM_WORKERS = 12
PREFETCH_FACTOR = 4
EPOCHS = 3                    # paper-faithful
LR = 3.0e-4                   # LoRA-friendly (50× higher than full FT)
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.05
TEMP = 0.07                   # ITC temperature

# LHP probabilistic sampling: r ~ N(0.5, 1+δ), δ∈[0,6]
LHP_DELTA = 3.0               # mid-range; paper uses δ scheduled across epochs
LHP_LOCAL_PROB_STD = 1.0 + LHP_DELTA

# LoRA config — only Q, K, V projection of MHA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_PATTERNS = ['q_proj', 'k_proj', 'v_proj', 'in_proj']  # PE-G14 uses in_proj for fused QKV in some layers

USE_BF16 = True
USE_COMPILE = False           # peft + compile có thể conflict; tắt cho an toàn
CHANNELS_LAST = True

MAX_TRAIN_ROWS = None         # set int để smoke test
SMOKE_TEST = False
VAL_EVERY_EPOCH = True
print('Config done.')

In [ ]:
# === Manifest + pose NPZ index ===
train_df = pd.read_parquet(MANIFEST_DIR / 'train_manifest_trainable.parquet')
val_df = pd.read_parquet(MANIFEST_DIR / 'val_split.parquet')
gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')
val_zs_df = pd.read_parquet(MANIFEST_DIR / 'val_zeroshot_scene.parquet')

def remap_path(p):
    if not isinstance(p, str) or not p:
        return p
    if p.startswith(str(INPUT_ROOT)):
        return p
    for anchor in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(anchor)
        if i >= 0:
            return str(INPUT_ROOT / p[i:])
    return p

for df_ in (train_df, val_df, gallery_df, val_zs_df):
    if 'image_path' in df_.columns:
        df_['image_path'] = df_['image_path'].map(remap_path)

if SMOKE_TEST:
    train_df = train_df.head(2000).copy()
if MAX_TRAIN_ROWS:
    train_df = train_df.head(MAX_TRAIN_ROWS).copy()

print('train:', train_df.shape, 'val:', val_df.shape, 'val_zs:', val_zs_df.shape)

# === Build pose keypoint lookup (image_id → 17×3 keypoints) ===
#
# Pose NPZ chunks là từ 01b. Load tất cả → dict by image_id.
POSE_DIR = OUTPUT_ROOT / 'pose' / 'vitposepp'
pose_kp_by_id = {}
for chunk in sorted((POSE_DIR / 'train').glob('chunk_*.npz')) if (POSE_DIR / 'train').exists() else []:
    z = np.load(chunk, allow_pickle=True)
    for _id, kp, ok in zip(z['ids'], z['keypoints'], z['valid']):
        if bool(ok):
            pose_kp_by_id[str(_id)] = np.asarray(kp, dtype=np.float32)
print(f'Loaded keypoints for {len(pose_kp_by_id):,} train images')

In [ ]:
# === Load PE-Core-G14-448, freeze backbone, inject LoRA ===
maybe_clear_cuda()
pe_model = pe.CLIP.from_config(PE_MODEL_NAME, pretrained=True).to(device).eval()

# Freeze 100%
for p in pe_model.parameters():
    p.requires_grad = False

# Inspect module names to map target_modules accurately
print('--- Inspecting PE-G14 attention modules (first 5 matches) ---')
attn_names = []
for name, mod in pe_model.named_modules():
    if any(t in name for t in ('attn', 'attention', '.qkv', '.in_proj', '.q_proj', '.k_proj', '.v_proj')):
        if any(p in name.split('.')[-1] for p in ('q_proj', 'k_proj', 'v_proj', 'in_proj', 'qkv')):
            attn_names.append(name)
for n in attn_names[:10]:
    print(' ', n)
print(f'... total {len(attn_names)} attention proj modules found')

# Pick targets actually present
actual_targets = set()
for n in attn_names:
    leaf = n.rsplit('.', 1)[-1]
    if leaf in ('q_proj', 'k_proj', 'v_proj', 'in_proj'):
        actual_targets.add(leaf)
    # PE-G14 may use 'qkv' fused projection — wrap as target too
    if leaf == 'qkv':
        actual_targets.add('qkv')
actual_targets = sorted(actual_targets)
if not actual_targets:
    raise RuntimeError('No q/k/v projection modules detected — inspect model arch.')
print('LoRA target modules:', actual_targets)

from peft import LoraConfig, get_peft_model, TaskType
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=actual_targets,
    task_type=TaskType.FEATURE_EXTRACTION,
)
pe_model = get_peft_model(pe_model, lora_cfg)
pe_model.print_trainable_parameters()

# Projection head 1280 → 768 (extra trainable, ~1M params)
emb_dim = getattr(pe_model.base_model.model, 'embed_dim', None) or 1280
proj_head = nn.Linear(emb_dim, 768, bias=False).to(device)
for p in proj_head.parameters():
    p.requires_grad = True

image_size = pe_model.base_model.model.image_size
pe_preprocess = pe_transforms.get_image_transform(image_size)
pe_tokenizer = pe_transforms.get_text_tokenizer(pe_model.base_model.model.context_length)
print(f'image_size={image_size} emb_dim={emb_dim}')

In [ ]:
# === LHP transform: local (anatomical crop) vs global (full image) ===
import torchvision.transforms.functional as TF
from torchvision import transforms as T

_global_aug = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
])

def _anatomical_crop(pil_img, kp_xyc):
    """kp_xyc: (17,3) normalized x,y in [0,1] + confidence.

    Choose a body region (torso / lower / face) randomly weighted by
    keypoint confidence, then crop and resize.
    """
    w, h = pil_img.size
    # COCO indices: 0 nose, 5 lsh, 6 rsh, 11 lhip, 12 rhip, 13 lkn, 14 rkn
    parts = {
        'face':  [0, 1, 2, 3, 4],
        'torso': [5, 6, 11, 12],
        'lower': [11, 12, 13, 14, 15, 16],
    }
    options = []
    for k, idxs in parts.items():
        sel = kp_xyc[idxs]
        if sel[:, 2].sum() > 0.5 and (sel[:, :2].sum(axis=1) > 0).any():
            options.append(k)
    if not options:
        return _global_aug(pil_img)
    chosen = options[np.random.randint(len(options))]
    pts = kp_xyc[parts[chosen]]
    mask = pts[:, 2] > 0
    if not mask.any():
        return _global_aug(pil_img)
    xs = pts[mask, 0] * w
    ys = pts[mask, 1] * h
    x0, x1 = float(xs.min()), float(xs.max())
    y0, y1 = float(ys.min()), float(ys.max())
    pad = 0.2 * max(x1 - x0, y1 - y0, 16)
    x0 = max(0, int(x0 - pad)); y0 = max(0, int(y0 - pad))
    x1 = min(w, int(x1 + pad)); y1 = min(h, int(y1 + pad))
    if x1 - x0 < 16 or y1 - y0 < 16:
        return _global_aug(pil_img)
    cropped = pil_img.crop((x0, y0, x1, y1))
    return T.Resize((image_size, image_size))(cropped)

class LHPDataset(Dataset):
    def __init__(self, df, kp_lookup, preprocess, tokenizer, lhp_std=LHP_LOCAL_PROB_STD):
        self.df = df.reset_index(drop=True)
        self.kp = kp_lookup
        self.preprocess = preprocess
        self.tokenizer = tokenizer
        self.lhp_std = lhp_std
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = row['image_path']
        caption = str(row['caption']) if pd.notna(row['caption']) else ''
        try:
            pil = Image.open(path).convert('RGB')
        except Exception:
            pil = Image.new('RGB', (image_size, image_size))
        # LHP probabilistic sampling
        r = np.random.normal(0.5, self.lhp_std)
        if r > 0.5 and str(row['image_id']) in self.kp:
            view = _anatomical_crop(pil, self.kp[str(row['image_id'])])
        else:
            view = _global_aug(pil)
        img_t = self.preprocess(view)
        return img_t, caption

def _collate(batch):
    imgs = torch.stack([b[0] for b in batch], dim=0)
    texts = [b[1] for b in batch]
    return imgs, texts

print('LHP transform ready.')

In [ ]:
# === ITC training loop ===
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

trainable = [p for p in pe_model.parameters() if p.requires_grad] + list(proj_head.parameters())
opt = AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
N_total = len(train_df)
steps_per_epoch = math.ceil(N_total / BATCH)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
sched = SequentialLR(
    opt,
    schedulers=[
        LinearLR(opt, start_factor=1e-3, end_factor=1.0, total_iters=warmup_steps),
        CosineAnnealingLR(opt, T_max=max(1, total_steps - warmup_steps), eta_min=1e-6),
    ],
    milestones=[warmup_steps],
)
scaler = GradScaler(enabled=(not USE_BF16))  # BF16 không cần scaler

logit_scale = nn.Parameter(torch.ones([]) * np.log(1.0 / TEMP), requires_grad=False).to(device)

train_ds = LHPDataset(train_df, pose_kp_by_id, pe_preprocess, pe_tokenizer)
train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    collate_fn=_collate, drop_last=True,
)

amp_dtype = torch.bfloat16 if USE_BF16 else torch.float16

# CRITICAL: val evaluation MUST pipe through proj_head exactly như training,
# nếu không LoRA weights bị orphan (chúng được train để hoạt động cùng proj_head).
@torch.inference_mode()
def _encode_image_set(df, id_col):
    """Encode image set with LoRA-merged model + proj_head; returns L2-norm 768-d fp16."""
    embs, ids = [], []
    pe_model.eval(); proj_head.eval()
    for start in tqdm(range(0, len(df), 64), desc='encode-eval'):
        rows = df.iloc[start:start + 64]
        imgs = []
        for p in rows['image_path']:
            try:
                pil = Image.open(p).convert('RGB')
            except Exception:
                pil = Image.new('RGB', (image_size, image_size))
            imgs.append(pe_preprocess(pil))
        x = torch.stack(imgs, 0).to(device, non_blocking=True)
        if CHANNELS_LAST:
            x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            f = pe_model.base_model.model.encode_image(x)
            f = proj_head(f)                          # ← pipe through proj_head
        f = F.normalize(f.float(), dim=-1).cpu().numpy().astype('float16')
        embs.append(f)
        ids.extend(rows[id_col].astype(str).tolist())
    return np.concatenate(embs, 0), np.array(ids)

@torch.inference_mode()
def _encode_text_set(texts):
    """Encode text list via base_model.encode_text + proj_head."""
    embs = []
    pe_model.eval(); proj_head.eval()
    for s in range(0, len(texts), 64):
        tk = pe_tokenizer(list(texts[s:s + 64])).to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            t = pe_model.base_model.model.encode_text(tk)
            t = proj_head(t)                          # ← pipe through proj_head
        embs.append(F.normalize(t.float(), dim=-1).cpu().numpy().astype('float16'))
    return np.concatenate(embs, 0)

best_val_top1 = -1.0
for epoch in range(EPOCHS):
    pe_model.train(); proj_head.train()
    pbar = tqdm(train_loader, desc=f'LHP-LoRA epoch {epoch+1}/{EPOCHS}')
    for step, (imgs, texts) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        if CHANNELS_LAST:
            imgs = imgs.contiguous(memory_format=torch.channels_last)
        tokens = pe_tokenizer(texts).to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            img_f = pe_model.base_model.model.encode_image(imgs)
            txt_f = pe_model.base_model.model.encode_text(tokens)
            img_f = F.normalize(proj_head(img_f), dim=-1)
            txt_f = F.normalize(proj_head(txt_f), dim=-1)
            logits_per_image = logit_scale.exp() * img_f @ txt_f.t()
            labels = torch.arange(imgs.size(0), device=device)
            loss_i2t = F.cross_entropy(logits_per_image, labels)
            loss_t2i = F.cross_entropy(logits_per_image.t(), labels)
            loss = 0.5 * (loss_i2t + loss_t2i)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
        if step % 20 == 0:
            pbar.set_postfix(loss=float(loss), lr=opt.param_groups[0]['lr'])

    # === Validation: encode val_split images + captions ===
    # Compute BOTH val_top1 AND val_mAP@10 (mAP@10 sát với AIC leaderboard hơn).
    if VAL_EVERY_EPOCH and len(val_df) > 0:
        sub = val_df.sample(n=min(2000, len(val_df)), random_state=42)
        val_imgs, val_ids = _encode_image_set(sub, 'image_id')
        val_txts = _encode_text_set(sub['caption'].astype(str).tolist())
        sims = val_txts.astype('float32') @ val_imgs.astype('float32').T   # (N, N) since single positive per query
        gt = np.arange(len(sims))                                          # diagonal positive
        ranks = (-sims).argsort(axis=1)
        ranks_of_gt = np.array([np.where(ranks[i] == gt[i])[0][0] for i in range(len(gt))])
        top1 = float((ranks_of_gt == 0).mean())
        # mAP@10: single-positive → AP = 1/(rank+1) if rank < 10 else 0
        ap10 = np.where(ranks_of_gt < 10, 1.0 / (ranks_of_gt + 1), 0.0)
        map10 = float(ap10.mean())
        print(f'[epoch {epoch+1}] val_top1={top1:.4f}  val_mAP@10={map10:.4f}  (n={len(sub)})')
        # Use mAP@10 as the selection metric (closer to AIC leaderboard)
        if map10 > best_val_top1:
            best_val_top1 = map10
            best_dir = LHP_CKPT_DIR / 'lora_best'
            pe_model.save_pretrained(best_dir)
            torch.save(proj_head.state_dict(), best_dir / 'proj_head.pt')
            print(f'  ↑ best updated: mAP@10={map10:.4f} → {best_dir}')
            # Drive sync
            drive_dst = DRIVE_OUTPUT_ROOT / 'checkpoints' / 'lhp_peg14' / 'lora_best'
            drive_dst.mkdir(parents=True, exist_ok=True)
            for src in best_dir.iterdir():
                shutil.copy2(src, drive_dst / src.name)

    # Per-epoch snapshot too
    ep_dir = LHP_CKPT_DIR / f'lora_epoch_{epoch+1}'
    pe_model.save_pretrained(ep_dir)
    torch.save(proj_head.state_dict(), ep_dir / 'proj_head.pt')

print(f'LHP-LoRA training done. best_val_mAP@10 = {best_val_top1:.4f}')


In [ ]:
# === Fallback: if LoRA val_mAP@10 < zero-shot mAP@10, use zero-shot ===
#
# Apples-to-apples comparison: both compute mAP@10 from val_text @ val_image cosine.
# best_val_top1 (despite name) holds mAP@10 of LoRA model (updated trong cell-train).
zs_val_npz = (OUTPUT_ROOT / 'features' / 'pe_g14' / 'val.npz')
zs_val_text_npz = (OUTPUT_ROOT / 'features' / 'pe_g14' / 'val_text.npz')
zs_map10 = None
if zs_val_npz.exists() and zs_val_text_npz.exists():
    z_img = np.load(zs_val_npz)
    z_txt = np.load(zs_val_text_npz)
    img_emb = z_img['embeddings'].astype('float32')
    txt_emb = z_txt['embeddings'].astype('float32')
    # Align by id (val img and val text share image_id 1-to-1)
    img_id2idx = {str(x): i for i, x in enumerate(z_img['ids'])}
    aligned_img, aligned_txt = [], []
    for ti, tid in enumerate(z_txt['ids']):
        if str(tid) in img_id2idx:
            aligned_img.append(img_emb[img_id2idx[str(tid)]])
            aligned_txt.append(txt_emb[ti])
    if aligned_img:
        aligned_img = np.stack(aligned_img); aligned_txt = np.stack(aligned_txt)
        # L2-norm
        aligned_img = aligned_img / np.maximum(np.linalg.norm(aligned_img, axis=1, keepdims=True), 1e-12)
        aligned_txt = aligned_txt / np.maximum(np.linalg.norm(aligned_txt, axis=1, keepdims=True), 1e-12)
        sims = aligned_txt @ aligned_img.T
        gt = np.arange(len(sims))
        ranks = (-sims).argsort(axis=1)
        ranks_of_gt = np.array([np.where(ranks[i] == gt[i])[0][0] for i in range(len(gt))])
        zs_top1 = float((ranks_of_gt == 0).mean())
        ap10 = np.where(ranks_of_gt < 10, 1.0 / (ranks_of_gt + 1), 0.0)
        zs_map10 = float(ap10.mean())
        print(f'zero-shot PE-G14: val_top1={zs_top1:.4f}  val_mAP@10={zs_map10:.4f}')

use_lora = True
lora_map10 = best_val_top1  # rename for clarity — already mAP@10 sau fix Fix 1
if zs_map10 is not None and lora_map10 < zs_map10:
    use_lora = False
    print(f'⚠️  LHP-LoRA mAP@10 {lora_map10:.4f} < zero-shot {zs_map10:.4f}. Fallback to zero-shot.')
else:
    print(f'LHP-LoRA wins: mAP@10 {lora_map10:.4f} ≥ zero-shot {zs_map10 if zs_map10 else "n/a"}. Use LoRA adapter.')

with open(LHP_CKPT_DIR / 'fallback_decision.json', 'w') as f:
    json.dump({
        'use_lora': use_lora,
        'lora_mAP@10': lora_map10,
        'zs_mAP@10': zs_map10,
        'note': 'mAP@10 single-positive: AP = 1/(rank+1) if rank<10 else 0',
    }, f, indent=2)


In [ ]:
# === Inference: encode gallery + queries + val + val_zs với merged LoRA + proj_head ===
#
# CRITICAL FIX: proj_head MUST be applied at inference vì LoRA weights được
# train kèm với proj_head. Bỏ qua proj_head → LoRA weights vô nghĩa.
#
# Trường hợp fallback (use_lora=False, zero-shot PE-G14):
#   PE-G14 raw output 1280-d không match dim với proj_head 1280→768.
#   → Bỏ proj_head luôn cho zero-shot, dùng raw 1280-d cosine.
from peft import PeftModel

if use_lora:
    # Merge LoRA into base for faster inference
    pe_model = pe_model.merge_and_unload()
    # Load best proj_head if available (in case current proj_head was overwritten by training noise)
    best_proj_path = LHP_CKPT_DIR / 'lora_best' / 'proj_head.pt'
    if best_proj_path.exists():
        proj_head.load_state_dict(torch.load(best_proj_path, map_location=device))
    proj_head.eval()
    print('LoRA merged into base + proj_head loaded for inference (output dim = 768).')
else:
    # Strip LoRA → vanilla PE-G14 zero-shot. proj_head random-init, không có
    # ý nghĩa gì → DISABLE proj_head và dùng raw 1280-d.
    pe_model = pe_model.base_model.model
    proj_head = None
    print('Using zero-shot PE-G14 (LoRA + proj_head disabled, output dim = 1280).')

pe_model.eval()

@torch.inference_mode()
def encode_images(df, id_col, batch=128):
    ids, paths, ok_arr, feats = [], [], [], []
    for start in tqdm(range(0, len(df), batch), desc=f'enc-img {id_col}'):
        rows = df.iloc[start:start + batch]
        imgs, oks = [], []
        for p in rows['image_path']:
            try:
                pil = Image.open(p).convert('RGB'); oks.append(True)
            except Exception:
                pil = Image.new('RGB', (image_size, image_size)); oks.append(False)
            imgs.append(pe_preprocess(pil))
        x = torch.stack(imgs, 0).to(device, non_blocking=True)
        if CHANNELS_LAST: x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            f = pe_model.encode_image(x)
            if proj_head is not None:
                f = proj_head(f)                       # ← apply proj_head
        f = F.normalize(f.float(), dim=-1).cpu().numpy().astype('float16')
        ids.extend(rows[id_col].astype(str).tolist())
        paths.extend(rows['image_path'].tolist() if 'image_path' in rows.columns else [''] * len(rows))
        feats.append(f); ok_arr.extend(oks)
    return np.concatenate(feats, 0), np.array(ids), np.array(paths), np.array(ok_arr)

@torch.inference_mode()
def encode_texts(ids, texts, batch=256):
    feats = []
    for s in tqdm(range(0, len(texts), batch), desc='enc-txt'):
        tk = pe_tokenizer(list(texts[s:s + batch])).to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            t = pe_model.encode_text(tk)
            if proj_head is not None:
                t = proj_head(t)                       # ← apply proj_head
        feats.append(F.normalize(t.float(), dim=-1).cpu().numpy().astype('float16'))
    return np.concatenate(feats, 0), np.array([str(x) for x in ids])

# Gallery + queries
g_emb, g_ids, g_paths, g_ok = encode_images(gallery_df, 'gallery_id')
q_emb, q_ids = encode_texts(query_df['query_index'].astype(str).tolist(), query_df['caption'].astype(str).tolist())
save_npz_atomic(LHP_FEAT_DIR / 'gallery.npz', ids=g_ids, paths=g_paths, embeddings=g_emb, ok=g_ok)
save_npz_atomic(LHP_FEAT_DIR / 'queries.npz', ids=q_ids, embeddings=q_emb)
print(f'Saved gallery {g_emb.shape}, queries {q_emb.shape}')

# Val + val_zs
for name, df in [('val', val_df), ('val_zs', val_zs_df)]:
    if len(df) == 0: continue
    v_img_emb, v_ids, v_paths, v_ok = encode_images(df, 'image_id')
    save_npz_atomic(LHP_FEAT_DIR / f'{name}.npz', ids=v_ids, paths=v_paths, embeddings=v_img_emb, ok=v_ok)
    v_txt_emb, _ = encode_texts(df['image_id'].astype(str).tolist(), df['caption'].astype(str).tolist())
    save_npz_atomic(LHP_FEAT_DIR / f'{name}_text.npz', ids=v_ids, embeddings=v_txt_emb)

# === Score matrix Q×G ===
Q = torch.from_numpy(q_emb.astype('float32')).to(device)
G = torch.from_numpy(g_emb.astype('float32')).to(device)
Q = F.normalize(Q, dim=-1); G = F.normalize(G, dim=-1)
S = (Q @ G.T).half().cpu()
torch.save({'scores': S, 'query_ids': q_ids, 'gallery_ids': g_ids,
            'use_lora': use_lora, 'output_dim': int(g_emb.shape[1])},
           LHP_FEAT_DIR / 'scores_lhp.pt')
print(f'Saved scores_lhp.pt shape={tuple(S.shape)} dim={g_emb.shape[1]} use_lora={use_lora}')

# Val score matrices (cho 09 adaptive ensemble val gate)
for name in ('val', 'val_zs'):
    img_p = LHP_FEAT_DIR / f'{name}.npz'
    txt_p = LHP_FEAT_DIR / f'{name}_text.npz'
    if not (img_p.exists() and txt_p.exists()): continue
    zi = np.load(img_p); zt = np.load(txt_p)
    Qv = F.normalize(torch.from_numpy(zt['embeddings'].astype('float32')).to(device), dim=-1)
    Gv = F.normalize(torch.from_numpy(zi['embeddings'].astype('float32')).to(device), dim=-1)
    Sv = (Qv @ Gv.T).half().cpu()
    torch.save({'scores': Sv, 'query_ids': zt['ids'], 'gallery_ids': zi['ids']},
               LHP_FEAT_DIR / f'scores_{name}.pt')
    print(f'Saved scores_{name}.pt shape={tuple(Sv.shape)}')

# Drive sync
for f in LHP_FEAT_DIR.rglob('*'):
    if f.is_file():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print('LHP-PE-G14 done.')
